# Getting started

In this notebook, we illustrate how to use the Neural News Recommendation with Multi-Head Self-Attention ([NRMS](https://aclanthology.org/D19-1671/)). The implementation is taken from the [recommenders](https://github.com/recommenders-team/recommenders) repository. We have simply stripped the model to keep it cleaner.

We use a small dataset, which is downloaded from [recsys.eb.dk](https://recsys.eb.dk/). All the datasets are stored in the folder path ```~/ebnerd_data/*```.

## Load functionality

In [ ]:
!pip install "transformers>=4.30.0,<4.37.3"
!pip install "tensorflow>=2.12.0,<2.16.0"
!pip install "torch>=2.0.0,<2.3.0"
!pip install "scikit-learn==1.4.0"
!pip install "numpy>=1.24.0,<1.26.1"
!pip install "polars==0.20.8"
!pip install "pyyaml==6.0.1"
!pip install tqdm

In [ ]:
# import zipfile
# import os

# def unzip_folder(zip_file_path, extract_to_dir):
#     if not os.path.exists(extract_to_dir):
#         os.makedirs(extract_to_dir)

#     with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_to_dir)

# zip_file_path = 'ebrec.zip'
# extract_to_dir = '.'
# unzip_folder(zip_file_path, extract_to_dir)

# zip_file_path = 'ebnerd_data.zip'
# extract_to_dir = '.'
# unzip_folder(zip_file_path, extract_to_dir)

In [ ]:
# import sys
# sys.path.append('.')

In [ ]:
from transformers import AutoTokenizer, AutoModel
from pathlib import Path
import tensorflow as tf
import polars as pl

from ebrec.utils._constants import (
    DEFAULT_HISTORY_ARTICLE_ID_COL,
    DEFAULT_CLICKED_ARTICLES_COL,
    DEFAULT_INVIEW_ARTICLES_COL,
    DEFAULT_IMPRESSION_ID_COL,
    DEFAULT_SUBTITLE_COL,
    DEFAULT_LABELS_COL,
    DEFAULT_TITLE_COL,
    DEFAULT_USER_COL,
)

from ebrec.utils._behaviors import (
    create_binary_labels_column,
    sampling_strategy_wu2019,
    add_known_user_column,
    add_prediction_scores,
    truncate_history,
)
from ebrec.evaluation import MetricEvaluator, AucScore, NdcgScore, MrrScore
from ebrec.utils._articles import convert_text2encoding_with_transformers
from ebrec.utils._polars import concat_str_columns, slice_join_dataframes
from ebrec.utils._articles import create_article_id_to_value_mapping
from ebrec.utils._nlp import get_transformers_word_embeddings
from ebrec.utils._python import write_submission_file, rank_predictions_by_score

from ebrec.models.newsrec.dataloader import NRMSDataLoader
from ebrec.models.newsrec.model_config import hparams_nrms
from ebrec.models.newsrec.nrms import NRMSModel

## Load dataset

In [ ]:
def ebnerd_from_path(path: Path, history_size: int = 30) -> pl.DataFrame:
    """
    Load ebnerd - function
    """
    df_history = (
        pl.scan_parquet(path.joinpath("history.parquet"))
        .select(DEFAULT_USER_COL, DEFAULT_HISTORY_ARTICLE_ID_COL)
        .pipe(
            truncate_history,
            column=DEFAULT_HISTORY_ARTICLE_ID_COL,
            history_size=history_size,
            padding_value=0,
            enable_warning=False,
        )
    )
    df_behaviors = (
        pl.scan_parquet(path.joinpath("behaviors.parquet"))
        .collect()
        .pipe(
            slice_join_dataframes,
            df2=df_history.collect(),
            on=DEFAULT_USER_COL,
            how="left",
        )
    )
    return df_behaviors

### Generate labels
We sample a few just to get started. For testset we just make up a dummy column with 0 and 1 - this is not the true labels.

In [ ]:
PATH = Path("ebnerd_data")
DATASPLIT = "ebnerd_small"

In this example we sample the dataset, just to keep it smaller. Also, one can simply add the testset similary to the validation.

In [ ]:
COLUMNS = [
    DEFAULT_USER_COL,
    DEFAULT_HISTORY_ARTICLE_ID_COL,
    DEFAULT_INVIEW_ARTICLES_COL,
    DEFAULT_CLICKED_ARTICLES_COL,
    DEFAULT_IMPRESSION_ID_COL,
    "read_time"
]
HISTORY_SIZE = 10
FRACTION = 0.01

df_train = (
    ebnerd_from_path(PATH.joinpath(DATASPLIT, "train"), history_size=HISTORY_SIZE)
    .select(COLUMNS)
    .pipe(
        sampling_strategy_wu2019,
        npratio=4,
        shuffle=True,
        with_replacement=True,
        seed=123,
    )
    .pipe(create_binary_labels_column)
    .sample(fraction=FRACTION)
)
# =>
df_validation = (
    ebnerd_from_path(PATH.joinpath(DATASPLIT, "validation"), history_size=HISTORY_SIZE)
    .select(COLUMNS)
    .pipe(create_binary_labels_column)
    .sample(fraction=FRACTION)
)
df_train.head(2)

## Load articles

In [ ]:
df_articles = pl.read_parquet(PATH.joinpath("articles.parquet"))
df_articles.head(2)

In [ ]:
df_articles_p = df_articles.to_pandas()
df_train_p = df_train.to_pandas()
df_validation_p = df_validation.to_pandas()

df_articles_p.to_csv('articles.csv', index=False)
df_train_p.to_csv('train.csv', index=False)
df_validation_p.to_csv('validation.csv', index=False)

In [ ]:
!pip install recommenders
!pip install --upgrade hyperopt

In [ ]:
import sys
import pandas as pd
import tensorflow as tf
import numpy as np
import random
tf.get_logger().setLevel('ERROR') # only show error messages

from recommenders.utils.timer import Timer
from recommenders.models.ncf.ncf_singlenode import NCF
from recommenders.models.ncf.dataset import Dataset as NCFDataset
from recommenders.datasets import movielens
from recommenders.datasets.python_splitters import python_chrono_split
from recommenders.evaluation.python_evaluation import (
    map, ndcg_at_k, precision_at_k, recall_at_k
)
from recommenders.utils.notebook_utils import store_metadata
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

# Specify the columns to keep
columns_to_keep = ['user_id', 'article_ids_clicked', 'read_time']

# Drop other columns in the training data
train_df_reduced = df_train_p[columns_to_keep]

# Drop other columns in the validation data
validation_df_reduced = df_validation_p[columns_to_keep]

# Drop rows with null values in the 'article_id' column
train_df_reduced = train_df_reduced.dropna(subset=['article_ids_clicked'])
validation_df_reduced = validation_df_reduced.dropna(subset=['article_ids_clicked'])

# Rename the columns
train_df_reduced.columns = ['userID', 'itemID', 'rating']
validation_df_reduced.columns = ['userID', 'itemID', 'rating']

# Convert 'itemID' and 'rating' columns to integer
# train_df_reduced['itemID'] = train_df_reduced['itemID'].astype(int)
# validation_df_reduced['itemID'] = validation_df_reduced['itemID'].astype(int)

# Convert 'itemID' strings to lists of integers and extract the first item
def to_int_list(item_string):
    int_list = [int(x) for x in item_string]
    return int_list[0] if int_list else None # Return the first item or None if list is empty

train_df_reduced['itemID'] = train_df_reduced['itemID'].apply(to_int_list)
validation_df_reduced['itemID'] = validation_df_reduced['itemID'].apply(to_int_list)

# Sort the training data by userID
train_df_reduced = train_df_reduced.sort_values(by='userID')
# Sort the validation data by userID
validation_df_reduced = validation_df_reduced.sort_values(by='userID')

validation_df_reduced = validation_df_reduced[validation_df_reduced["userID"].isin(validation_df_reduced["userID"].unique())]
validation_df_reduced = validation_df_reduced[validation_df_reduced["itemID"].isin(validation_df_reduced["itemID"].unique())]

# Print the updated dataframes to verify
print("Training Data (Reduced):")
print(train_df_reduced.head())

print("\nValidation Data (Reduced):")
print(validation_df_reduced.head())

# Merge train and validation data with articles
train_df_merged = pd.merge(train_df_reduced, df_articles_p, left_on='itemID', right_on='article_id', how='inner')
validation_df_merged = pd.merge(validation_df_reduced, df_articles_p, left_on='itemID', right_on='article_id', how='inner')

# Specify the columns to keep
columns_to_keep = ['userID', 'itemID', 'premium', 'article_type', 'category_str', 'sentiment_label', 'rating']

# Drop other columns in the training data
train_df_merged = train_df_merged[columns_to_keep]

# Drop other columns in the validation data
validation_df_merged = validation_df_merged[columns_to_keep]

# Rename the columns
train_df_merged.columns = ['userID', 'itemID', 'premium', 'article_type', 'category_str', 'sentiment_label', 'rating']
validation_df_merged.columns = ['userID', 'itemID', 'premium', 'article_type', 'category_str', 'sentiment_label', 'rating']

# Sort the training data by userID
train_df_merged = train_df_reduced.sort_values(by='userID')
# Sort the validation data by userID
validation_df_merged = validation_df_reduced.sort_values(by='userID')

train_df_merged.to_csv('train.csv', index=False)
validation_df_merged.to_csv('validation.csv', index=False)

train_file = "train.csv"
test_file = "validation.csv"

TOP_K = 5

EPOCHS = 50
BATCH_SIZE = 256

SEED = 42

data = NCFDataset(train_file=train_file, test_file=test_file, seed=SEED)

def objective(params):
    model = NCF(
        n_users=data.n_users,
        n_items=data.n_items,
        model_type="NeuMF",
        n_factors=int(params['n_factors']),
        layer_sizes=[int(params['layer_sizes1']), int(params['layer_sizes2']), int(params['layer_sizes3'])],
        n_epochs=int(params['n_epochs']),
        batch_size=int(params['batch_size']),
        learning_rate=params['learning_rate'],
        verbose=10,
        seed=SEED
    )

    with Timer() as train_time:
        model.fit(data)

    users, items, preds = [], [], []
    item = list(train_df_reduced.itemID.unique())
    for user in train_df_reduced.userID.unique():
        user = [user] * len(item)
        users.extend(user)
        items.extend(item)
        preds.extend(list(model.predict(user, item, is_list=True)))

    all_predictions = pd.DataFrame(data={"userID": users, "itemID": items, "prediction": preds})

    merged = pd.merge(train_df_reduced, all_predictions, on=["userID", "itemID"], how="outer")
    all_predictions = merged[merged.rating.isnull()].drop('rating', axis=1)

    eval_map = map(validation_df_reduced, all_predictions, col_prediction='prediction', k=TOP_K)
    eval_ndcg = ndcg_at_k(validation_df_reduced, all_predictions, col_prediction='prediction', k=TOP_K)
    eval_precision = precision_at_k(validation_df_reduced, all_predictions, col_prediction='prediction', k=TOP_K)
    eval_recall = recall_at_k(validation_df_reduced, all_predictions, col_prediction='prediction', k=TOP_K)
    print("Eval map for this use case is:{eval_map}")
    print(eval_map)
    print(params['n_factors'])
    print(params['layer_sizes1'])
    print(params['layer_sizes2'])
    print(params['layer_sizes3'])
    print(params['n_epochs'])
    print(params['batch_size'])
    print(params['learning_rate'])

    return {'loss': -eval_ndcg, 'status': STATUS_OK}

# Define the search space
space = {
    'n_factors': hp.quniform('n_factors', 4, 16, 1),
    'layer_sizes1': hp.quniform('layer_sizes1', 8, 64, 1),
    'layer_sizes2': hp.quniform('layer_sizes2', 4, 32, 1),
    'layer_sizes3': hp.quniform('layer_sizes3', 2, 16, 1),
    'n_epochs': hp.quniform('n_epochs', 10, 100, 10),
    'batch_size': hp.quniform('batch_size', 32, 512, 32),
    'learning_rate': hp.loguniform('learning_rate', -4, -2)
}

# Perform Bayesian Optimization
trials = Trials()
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=50,
            trials=trials,
            rstate=np.random.default_rng(SEED))

print("Best hyperparameters found:")
print(best)

In [ ]:
# import pandas as pd
# from sklearn.metrics.pairwise import cosine_similarity

# # Load datasets (replace with your actual data loading)
# df_train_p = df_train.to_pandas()
# df_validation_p = df_validation.to_pandas()

# # Prepare data
# user_column = 'user_id'
# inview_column = 'article_ids_inview'
# clicked_column = 'article_ids_clicked'

# # Explode the 'article_ids_inview' column to have one row per item interaction
# df_train_exploded = df_train_p.explode(inview_column)

# # De-duplicate the exploded DataFrame based on user and item before pivoting
# df_train_exploded_deduplicated = df_train_exploded.drop_duplicates(subset=[user_column, inview_column])

# # Create user-item interaction matrix (fill missing values with 0)
# # Here, we assume clicks are implicit positive feedback (1) and non-clicks are 0
# interaction_matrix = df_train_exploded_deduplicated.pivot(index=user_column, columns=inview_column, values=clicked_column).fillna(0)

# # Convert all values in the interaction matrix to numeric type (assuming clicked_column contains item IDs that need to be treated numerically)
# interaction_matrix = interaction_matrix.apply(pd.to_numeric, errors='coerce').fillna(0)

# # Calculate item-item cosine similarity
# item_similarity = cosine_similarity(interaction_matrix.T)

# def predict_clicked_item(user_id, inview_items):
#     if user_id not in interaction_matrix.index:
#         return None  # Handle cases where user is not in training data

#     # Filter items in interaction matrix to only those in the inview list
#     relevant_items = [item for item in inview_items if item in interaction_matrix.columns]

#     if not relevant_items:
#         return None  # No items in inview list are present in the training data

#     predicted_scores = []
#     for item_id in relevant_items:
#         similarities = item_similarity[interaction_matrix.columns.get_loc(item_id)]
#         user_ratings = interaction_matrix.loc[user_id]

#         weighted_sum = np.dot(similarities, user_ratings)
#         sum_of_weights = np.sum(np.abs(similarities))

#         if sum_of_weights != 0:
#             score = weighted_sum / sum_of_weights
#         else:
#             score = 0  # Handle cases where no similar items were interacted with by the user

#         predicted_scores.append((item_id, score))

#     # Sort items by predicted score and return the one with the highest score
#     predicted_scores.sort(key=lambda x: x[1], reverse=True)
#     return predicted_scores[0][0]

# # Example prediction
# user_id_to_predict = df_validation_p.iloc[6][user_column]
# inview_items_to_predict = df_validation_p.iloc[0][inview_column]  # Assuming 'article_ids_inview' contains lists
# predicted_item = predict_clicked_item(user_id_to_predict, inview_items_to_predict)

# # Iterate over rows and their index in the DataFrame
# for index, row in df_validation_p.head(10000).iterrows():
#     # Iterate over columns and their values in the current row
#     for col, value in row.items():
#       if(col == 'user_id'):
#         # print(f"element: {col} value: {value}")
#         inview_items_to_predict = df_validation_p.iloc[0][inview_column]  # Assuming 'article_ids_inview' contains lists
#         predicted_item = predict_clicked_item(user_id_to_predict, inview_items_to_predict)
#         if(predicted_item != None):
#           print(f"For user {user_id_to_predict} with in-view items {inview_items_to_predict}, predicted clicked item: {predicted_item}")

# # print(f"For user {user_id_to_predict} with in-view items {inview_items_to_predict}, predicted clicked item: {predicted_item}")

In [ ]:
# ncf = NCF (
#     n_users=data.n_users,
#     n_items=data.n_items,
#     model_type="NeuMF",
#     n_factors=4,
#     layer_sizes=[16,8,4],
#     n_epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     learning_rate=1e-3,
#     verbose=10,
#     seed=SEED
# )

# ncf.fit()

# MODEL_NAME = "NCF"
# LOG_DIR = f"downloads/runs/{MODEL_NAME}"
# MODEL_WEIGHTS = f"downloads/data/state_dict/{MODEL_NAME}/weights"

# # CALLBACKS
# tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=LOG_DIR, histogram_freq=1)
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2)
# modelcheckpoint = tf.keras.callbacks.ModelCheckpoint(
#     filepath=MODEL_WEIGHTS, save_best_only=True, save_weights_only=True, verbose=1
# )

# hparams_nrms.history_size = HISTORY_SIZE
# model = NRMSModel(
#     hparams=hparams_nrms,
#     word2vec_embedding=word2vec_embedding,
#     seed=42,
# )
# hist = model.model.fit(
#     train_dataloader,
#     validation_data=val_dataloader,
#     epochs=1,
#     callbacks=[tensorboard_callback, early_stopping, modelcheckpoint],
# )
# _ = model.model.load_weights(filepath=MODEL_WEIGHTS)